# PiShield — using a Shield Layer at training time

This notebook runs **end-to-end with no external downloads**. We train a tiny MLP whose
3-dimensional output must be **non-increasing** (`y_0 >= y_1 >= y_2`). By wrapping the model
with a Shield Layer, its outputs satisfy that requirement *at every step* — during training
and at inference. We compare against an unconstrained baseline to see the difference.

In [ ]:
import torch
import torch.nn as nn
from pishield.shield_layer import build_shield_layer

torch.manual_seed(0)

## 1. A tiny synthetic task

Random inputs, and non-increasing 3-vector targets. We use **small gaps** between consecutive
values, so the ordering is easy to get *wrong*: an unconstrained model will tend to invert
some of them, which is exactly what the requirement forbids.

In [ ]:
N, in_dim, out_dim = 512, 4, 3

X = torch.randn(N, in_dim)
base = torch.rand(N, 1)
gaps = torch.rand(N, 2) * 0.05  # small gaps -> ordering is non-trivial to preserve
Y = torch.cat([base, base - gaps[:, :1], base - gaps.sum(1, keepdim=True)], dim=1)
print("example target (non-increasing):", [round(v, 2) for v in Y[0].tolist()])

## 2. Write the requirements file (`y_0 >= y_1 >= y_2`)

In [ ]:
requirements_path = "monotonic_requirements.txt"
with open(requirements_path, "w") as f:
    f.write("ordering y_0 y_1 y_2\ny_0 - y_1 >= 0\ny_1 - y_2 >= 0\n")

## 3. Define the models

`ShieldedMLP` simply applies the Shield Layer to the MLP's output inside `forward`; that is
the *only* change compared to the plain baseline.

In [ ]:
def make_mlp():
    return nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, out_dim))

class ShieldedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = make_mlp()
        self.shield = build_shield_layer(out_dim, requirements_path)

    def forward(self, x):
        return self.shield(self.model(x))

## 4. Train both models and compare

We count how many outputs violate the requirement. The same training loop works for both,
because the Shield Layer is differentiable and gradients flow through it.

In [ ]:
def num_violations(out, tol=1e-6):
    bad = (out[:, 0] < out[:, 1] - tol) | (out[:, 1] < out[:, 2] - tol)
    return int(bad.sum())

def train(model, epochs=300, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        opt.step()
    return loss.item()

baseline = make_mlp()
shielded = ShieldedMLP()

base_mse = train(baseline)
shield_mse = train(shielded)

with torch.no_grad():
    base_out, shield_out = baseline(X), shielded(X)

print(f"baseline : final MSE={base_mse:.4f}, violations={num_violations(base_out)}/{N}")
print(f"shielded : final MSE={shield_mse:.4f}, violations={num_violations(shield_out)}/{N}")

## 5. Takeaway

The **shielded** model has **zero** violations by construction — the Shield Layer enforces
the requirement on every forward pass — while still learning the task at least as well
(here it even reaches a slightly lower MSE). The unconstrained baseline minimises the MSE too, but leaves many outputs violating
the requirement.